# Marsad — نوت بوك موحّد (Kaggle) — نسخة مُصلحة
يشغّل v2 عبر llama-server، ويبني مهام CVEfixes، ويدير حلقة Self-Play التدريجية، ويستقبل أوامر من Termux.

**الأسرار المطلوبة في Kaggle → Add-ons → Secrets** (لا تضع أي سر في الخلايا):
- `HF_TOKEN`: توكن دقيق الصلاحيات (كتابة على `marsad-endpoints` فقط + قراءة مستودع GGUF).
- `MARSAD_REF`: بصمة commit كاملة (40 حرفاً) **راجعتَها أنت**. لا يُشغَّل كود لم تراجعه.
- `GGUF_SHA256` (اختياري لكن موصى به): بصمة ملف النموذج؛ تُطبع أول مرة لتثبّتها.

In [ ]:
import hashlib, glob, json, os, re, secrets, shutil, subprocess, sys, time, urllib.request
from pathlib import Path

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub", "datasets"], check=True)
from kaggle_secrets import UserSecretsClient

_sec = UserSecretsClient()
def secret(name, required=True):
    try:
        v = _sec.get_secret(name)
    except Exception:
        v = ""
    if required and not v:
        raise RuntimeError(f"أضف السر {name} في Kaggle Secrets")
    return v

HF_TOKEN = secret("HF_TOKEN")      # يبقى متغيراً داخلياً: لا نضعه في os.environ كي لا ترثه العمليات الفرعية
MARSAD_REF = secret("MARSAD_REF")
assert re.fullmatch(r"[0-9a-f]{40}", MARSAD_REF), "MARSAD_REF يجب أن يكون SHA كاملاً (40 حرفاً)"

WORK = Path("/kaggle/working/Marsad")
if not WORK.exists():
    subprocess.run(["git", "clone", "--no-checkout", "https://github.com/AbuYahyaAlalwi2/Marsad.git", str(WORK)], check=True)
subprocess.run(["git", "-C", str(WORK), "fetch", "--depth", "1", "origin", MARSAD_REF], check=True)
subprocess.run(["git", "-C", str(WORK), "checkout", "--detach", MARSAD_REF], check=True)
head = subprocess.run(["git", "-C", str(WORK), "rev-parse", "HEAD"], capture_output=True, text=True, check=True).stdout.strip()
assert head == MARSAD_REF, "الكود المسحوب لا يطابق التعليق المثبَّت"
os.chdir(WORK)
print("✓ كود مثبَّت على", head[:12])

In [ ]:
from huggingface_hub import hf_hub_download

GGUF_REPO, GGUF_FILENAME = "AbuYahya2/deepseek-marsad-v2-gguf", "model-q8.gguf"
model_path = hf_hub_download(repo_id=GGUF_REPO, filename=GGUF_FILENAME, token=HF_TOKEN)

h = hashlib.sha256()
with open(model_path, "rb") as f:
    for chunk in iter(lambda: f.read(1 << 20), b""):
        h.update(chunk)
expected = secret("GGUF_SHA256", required=False)
if expected:
    assert h.hexdigest() == expected, "بصمة النموذج لا تطابق المتوقع: توقّف"
    print("✓ بصمة النموذج مطابقة")
else:
    print("⚠ ثبّت هذه البصمة في السر GGUF_SHA256:", h.hexdigest())

# الملف التنفيذي: الجاهز من dataset إن وُجد، وإلا بناء من المصدر (أبطأ)
bins = glob.glob("/kaggle/input/**/llama-server", recursive=True)
env = dict(os.environ)
if bins:
    bin_dir = Path("/kaggle/working/llama-bin"); bin_dir.mkdir(exist_ok=True)
    for f in Path(bins[0]).parent.iterdir():
        if f.is_file():
            shutil.copy2(f, bin_dir / f.name); os.chmod(bin_dir / f.name, 0o755)
    LLAMA = str(bin_dir / "llama-server")
    env["LD_LIBRARY_PATH"] = f"{bin_dir}:{env.get('LD_LIBRARY_PATH', '')}"
else:
    src = Path("/kaggle/working/llama.cpp")
    if not (src / "build/bin/llama-server").exists():
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/ggml-org/llama.cpp", str(src)], check=True)
        subprocess.run(["cmake", "-B", str(src / "build"), "-S", str(src), "-DGGML_CUDA=ON"], check=True)
        subprocess.run(["cmake", "--build", str(src / "build"), "--config", "Release", "-j", str(os.cpu_count())], check=True)
    LLAMA = str(src / "build/bin/llama-server")
env["GGML_CUDA_NO_PINNED"] = "1"
env.pop("HF_TOKEN", None); env.pop("HUGGING_FACE_HUB_TOKEN", None)

LLAMA_KEY = secrets.token_hex(24)
def start_server(extra):
    log = open("/kaggle/working/llama_server.log", "w")
    cmd = [LLAMA, "-m", model_path, "--host", "127.0.0.1", "--port", "8080", "-ngl", "99", "-c", "4096",
           "--parallel", "1", "--api-key", LLAMA_KEY] + extra
    return subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT, env=env)

def wait_ready(proc, seconds=420):
    for _ in range(seconds // 3):
        if proc.poll() is not None:
            return False
        try:
            if urllib.request.urlopen("http://127.0.0.1:8080/health", timeout=3).status == 200:
                return True
        except Exception:
            pass
        time.sleep(3)
    return False

server_proc = start_server(["--jinja"])
if not wait_ready(server_proc):
    server_proc.kill()
    server_proc = start_server([])          # ملف تنفيذي قديم لا يعرف --jinja
if not wait_ready(server_proc):
    print(Path("/kaggle/working/llama_server.log").read_text()[-3000:])
    raise RuntimeError("الخادم لم يبدأ")

# الوكيل يقرأ هذه المتغيرات (مفتاح الخادم المحلي فقط، لا مفتاح HF)
os.environ["MARSAD_LLM_URL"] = "http://127.0.0.1:8080/v1"
os.environ["MARSAD_LLM_KEY"] = LLAMA_KEY
os.environ["MARSAD_LLM_MODEL"] = "v2"
print("✓ الخادم جاهز على 127.0.0.1:8080 (محلي، بمفتاح)")

In [ ]:
sys.path.insert(0, str(WORK))
from datasets import load_dataset
from challenge.task_builder import build_tasks

ds = load_dataset("ASSERT-KTH/CVEfixes", split="train")
print("الأعمدة المتاحة فعلاً:", ds.column_names)
tasks, stats = build_tasks(ds.shuffle(seed=42), limit=300)
print("إحصاءات البناء:", stats)
assert tasks, "لم تُبنَ أي مهمة: راجع أسماء الأعمدة أعلاه"
OUT = WORK / "challenge" / "repo"; OUT.mkdir(parents=True, exist_ok=True)
(OUT / "tasks.json").write_text(json.dumps(tasks, ensure_ascii=False, indent=1), encoding="utf-8")
print("أعمدة الخطورة المستخدمة:", sorted({t["cvss_column"] for t in tasks}),
      "| تقييم:", sum(t["split"] == "eval" for t in tasks), "| تدريب:", sum(t["split"] == "train" for t in tasks))

In [ ]:
from huggingface_hub import HfApi, hf_hub_download

CONTROL_REPO = "AbuYahya2/marsad-endpoints"       # يجب أن يكون dataset خاصاً
ALLOWED = {"run", "pause", "stop"}
api = HfApi(token=HF_TOKEN)

def read_command(last="run"):
    """فشل الشبكة أو أمر غير معروف: نُبقي الحالة السابقة (لا نغيّر السلوك بصمت)."""
    try:
        p = hf_hub_download(CONTROL_REPO, "commands.json", repo_type="dataset", token=HF_TOKEN, force_download=True)
        a = json.loads(Path(p).read_text()).get("action")
        return a if a in ALLOWED else last
    except Exception:
        return last

def write_status(status):
    try:
        Path("/kaggle/working/status.json").write_text(json.dumps(status, ensure_ascii=False))
        api.upload_file(path_or_fileobj="/kaggle/working/status.json", path_in_repo="status.json",
                        repo_id=CONTROL_REPO, repo_type="dataset")
    except Exception as e:
        print("تعذر رفع الحالة:", type(e).__name__)

print("✓ قناة التحكم: أوامر مسموحة =", sorted(ALLOWED))

In [ ]:
from core.agent_loop import get_agent                       # يجب أن يقرأ MARSAD_LLM_URL/KEY/MODEL
from challenge.curriculum_self_play import CurriculumRunner
from challenge.judge_v2 import judge

runner = CurriculumRunner(get_agent(), judge, tasks, base_dir=WORK / "challenge", max_hours=11)
print("نهاية الحلقة:", runner.run(control=read_command, status=write_status))